# Healthcare Pipeline Testing Suite

This notebook contains 14 comprehensive tests to verify the complete batch tracking implementation across all pipeline layers.

## Test Sequence

1. **Control Table Initially Empty** - Verify starting state
2. **Batch Creation** - Test batch ID generation
3. **Stale Batch Recovery** - Verify FAILED status on stale batches
4. **Bronze Batch ID** - Verify batch propagation to Bronze layer
5. **Silver Batch ID** - Verify batch propagation to Silver layer
6. **Gold Batch ID** - Verify batch propagation to Gold layer
7. **Complete Lineage** - End-to-end batch ID consistency
8. **Success Status** - Verify COMPLETED status
9. **Failure Status** - Verify FAILED status handling
10. **Metadata** - Verify metadata columns
11. **File Inventory** - Verify file tracking
12. **Batch Consistency** - Verify single batch per execution
13. **Duplicate RUNNING Check** - Verify no duplicate RUNNING batches
14. **End-to-End Validation** - Complete validation query

## Instructions

⚠️ **Do not jump directly to the DLT pipeline. Test in sequence.**

## Test 1 – Control Table Initially Empty

**Purpose:** Verify the control table starting state.

**Expected Result:** No rows (or only historical rows if you didn't clear them).

In [0]:
%sql
SELECT *
FROM healthcare.control.pipeline_batches
ORDER BY start_time DESC;

## Test 2 – Batch Creation

**Purpose:** Test batch ID generation by running Task_1_Batch_ID_Generation.

**Instructions:** Run only Task_1_Batch_ID_Generation task.

**Expected Result:**
- `batch_id` is generated (format: YYYYMMDDHHMMSS)
- `status` = `RUNNING`
- `start_time` is populated
- `end_time` = `NULL`

In [0]:
%sql
SELECT *
FROM healthcare.control.pipeline_batches
ORDER BY start_time DESC;

## Test 3 – Stale Batch Recovery

**Purpose:** Verify that running Task_1_Batch_ID_Generation again marks the previous batch as FAILED.

**Instructions:** Run Task_1_Batch_ID_Generation again without running the pipeline.

**Expected Result:**
- **New Batch:** `status` = `RUNNING`
- **Old Batch:** `status` = `FAILED`
- ⚠️ **There must never be two RUNNING batches**

In [0]:
%sql
SELECT
  batch_id,
  status,
  start_time,
  end_time
FROM healthcare.control.pipeline_batches
ORDER BY start_time DESC;

## Test 4 – Bronze Batch ID

**Purpose:** Verify batch ID propagation to Bronze layer after pipeline finishes.

**Expected Result:** All Bronze tables should return exactly the same `batch_id` as in `pipeline_batches`.

In [0]:
%sql
-- Check Bronze Patients
SELECT DISTINCT batch_id
FROM healthcare.bronze.bronze_patients;

-- Check Bronze Staff
SELECT DISTINCT batch_id
FROM healthcare.bronze.bronze_staff;

-- Check Bronze Staff Schedule
SELECT DISTINCT batch_id
FROM healthcare.bronze.bronze_staff_schedule;

-- Check Bronze Services Weekly
SELECT DISTINCT batch_id
FROM healthcare.bronze.bronze_services_weekly;

## Test 5 – Silver Batch ID

**Purpose:** Verify batch ID propagation to Silver layer.

**Expected Result:**
- Exactly one batch ID
- Must match Bronze batch ID

In [0]:
%sql
SELECT DISTINCT batch_id FROM healthcare.silver.silver_patients;
SELECT DISTINCT batch_id FROM healthcare.silver.silver_staff;
SELECT DISTINCT batch_id FROM healthcare.silver.silver_staff_schedule;
SELECT DISTINCT batch_id FROM healthcare.silver.silver_services;


## Test 6 – Gold Batch ID

**Purpose:** Verify batch ID propagation to Gold layer.

**Expected Result:** Exactly one batch ID across all Gold tables.

In [0]:
%sql
-- Check Gold Patient Flow
SELECT DISTINCT batch_id
FROM healthcare.gold.patient_flow;

-- Check Gold Staff Efficiency
SELECT DISTINCT batch_id
FROM healthcare.gold.staff_efficiency;

-- Check Gold Bed Utilization
SELECT DISTINCT batch_id
FROM healthcare.gold.bed_utilization;

-- Check Gold Operational Efficiency
SELECT DISTINCT batch_id
FROM healthcare.gold.operational_efficiency;

-- Check Gold Bottleneck Analysis
SELECT DISTINCT batch_id
FROM healthcare.gold.bottleneck_analysis;

## Test 7 – Complete Lineage

**Purpose:** Verify batch ID consistency across all layers (Control → Bronze → Silver → Gold).

**Expected Result:** Every row should have the **same batch ID**.

In [0]:
%sql
SELECT
  'Bronze',
  MAX(batch_id)
FROM healthcare.bronze.bronze_patients

UNION ALL

SELECT
  'Silver',
  MAX(batch_id)
FROM healthcare.silver.silver_patients

UNION ALL

SELECT
  'Gold',
  MAX(batch_id)
FROM healthcare.gold.patient_flow;


## Test 8 – Success Status

**Purpose:** Verify that the batch status is updated to COMPLETED after successful workflow completion.

**Instructions:** Run after workflow completes successfully.

**Expected Result:** `status` = `COMPLETED`

In [0]:
%sql
SELECT
  batch_id,
  status,
  start_time,
  end_time
FROM healthcare.control.pipeline_batches
ORDER BY start_time DESC
LIMIT 1;

## Test 9 – Failure Status

**Purpose:** Verify that the batch status is updated to FAILED when the pipeline encounters an error.

**Instructions:**
1. Force the pipeline to fail (e.g., temporarily use an invalid S3 path in one Bronze table)
2. Run the workflow
3. Execute the query below

**Expected Result:** `status` = `FAILED`

In [0]:
%sql
SELECT
  batch_id,
  status,
  start_time,
  end_time
FROM healthcare.control.pipeline_batches
ORDER BY start_time DESC
LIMIT 1;

## Test 10 – Metadata

**Purpose:** Verify that metadata columns are populated correctly.

**Expected Result:**
- `batch_id` populated
- `pipeline_stage` = `BRONZE`
- `processed_timestamp` populated

In [0]:
%sql
SELECT
  batch_id,
  pipeline_stage,
  processed_timestamp
FROM healthcare.bronze.bronze_patients
LIMIT 20;

## Test 11 – File Inventory

**Purpose:** Verify that the file inventory table tracks files correctly.

**Expected Result:**
- Batch ID present
- Record counts populated
- Process status populated

In [0]:
%sql
SELECT
  file_name,
  batch_id,
  record_count,
  process_status
FROM healthcare.bronze.file_inventory;

## Test 12 – Batch Consistency

**Purpose:** Verify that each table contains data from exactly one batch per execution.

**Expected Result:** Count = `1` for a single pipeline execution.

In [0]:
%sql


-- Full batch consistency check across all layers
SELECT 'bronze_patients' AS table_name, COUNT(DISTINCT batch_id) AS batch_count
FROM healthcare.bronze.bronze_patients
UNION ALL
SELECT 'bronze_staff', COUNT(DISTINCT batch_id)
FROM healthcare.bronze.bronze_staff
UNION ALL
SELECT 'bronze_staff_schedule', COUNT(DISTINCT batch_id)
FROM healthcare.bronze.bronze_staff_schedule
UNION ALL
SELECT 'bronze_services_weekly', COUNT(DISTINCT batch_id)
FROM healthcare.bronze.bronze_services_weekly

UNION ALL

SELECT 'silver_patients', COUNT(DISTINCT batch_id)
FROM healthcare.silver.silver_patients
UNION ALL
SELECT 'silver_staff', COUNT(DISTINCT batch_id)
FROM healthcare.silver.silver_staff
UNION ALL
SELECT 'silver_staff_schedule', COUNT(DISTINCT batch_id)
FROM healthcare.silver.silver_staff_schedule
UNION ALL
SELECT 'silver_services', COUNT(DISTINCT batch_id)
FROM healthcare.silver.silver_services

UNION ALL

SELECT 'patient_flow', COUNT(DISTINCT batch_id)
FROM healthcare.gold.patient_flow
UNION ALL
SELECT 'staff_efficiency', COUNT(DISTINCT batch_id)
FROM healthcare.gold.staff_efficiency
UNION ALL
SELECT 'bed_utilization', COUNT(DISTINCT batch_id)
FROM healthcare.gold.bed_utilization
UNION ALL
SELECT 'operational_efficiency', COUNT(DISTINCT batch_id)
FROM healthcare.gold.operational_efficiency
UNION ALL
SELECT 'bottleneck_analysis', COUNT(DISTINCT batch_id)
FROM healthcare.gold.bottleneck_analysis;


## Test 13 – Duplicate RUNNING Check

**Purpose:** Verify that there is never more than one RUNNING batch at a time.

**Expected Result:**
- During execution: Count = `1`
- After completion: Count = `0`

In [0]:
%sql
-- Run during pipeline execution
SELECT
  COUNT(*) AS running_batch_count
FROM healthcare.control.pipeline_batches
WHERE status='RUNNING';

In [0]:
%sql
-- Run after pipeline completion
SELECT
  COUNT(*) AS running_batch_count
FROM healthcare.control.pipeline_batches
WHERE status='RUNNING';

## Test 14 – End-to-End Validation

**Purpose:** Complete validation joining control, Bronze, Silver, and Gold layers.

**Expected Result:**
- One row per pipeline execution
- Identical `batch_id` across control, Bronze, Silver, and Gold
- Status correctly set to `COMPLETED` or `FAILED`
- No duplicate RUNNING entries

In [0]:
%sql

SELECT
  pb.batch_id,
  pb.status,
  COUNT(DISTINCT bp.batch_id) AS bronze_batches,
  COUNT(DISTINCT sp.batch_id) AS silver_batches,
  COUNT(DISTINCT gp.batch_id) AS gold_batches
FROM healthcare.control.pipeline_batches pb
LEFT JOIN healthcare.bronze.bronze_patients bp
  ON pb.batch_id = bp.batch_id
LEFT JOIN healthcare.silver.silver_patients sp
  ON pb.batch_id = sp.batch_id
LEFT JOIN healthcare.gold.patient_flow gp
  ON pb.batch_id = gp.batch_id
GROUP BY pb.batch_id, pb.status, pb.start_time
ORDER BY pb.start_time DESC;


## ✅ Testing Complete

All 14 tests have been defined. Execute them in sequence to verify your pipeline implementation.

### Next Steps

1. Start with Test 1 to verify the initial state
2. Run Task_1_Batch_ID_Generation and execute Test 2
3. Continue through each test in order
4. Document any failures or discrepancies
5. Fix issues and re-test

### Success Criteria

✓ Batch ID propagates consistently across all layers  
✓ Status updates correctly (RUNNING → COMPLETED/FAILED)  
✓ No duplicate RUNNING batches  
✓ Metadata columns populated  
✓ File inventory tracks files correctly  
✓ End-to-end lineage is intact